### site_validation for 1D


In [6]:
import os
import numpy as np
from mt_metadata.transfer_functions import TF

def is_1d_site(edi_path, skew_threshold=0.2):
    """Check if an EDI file represents a 1D-like site using Swift skew."""
    try:
        edi_obj = TF()
        edi_obj.read(fn=edi_path)
        Z = np.array(edi_obj.impedance.data)
        freq = np.array(edi_obj.frequency)

        # Compute Swift skew
        Zxx, Zxy, Zyx, Zyy = Z[:,0,0], Z[:,0,1], Z[:,1,0], Z[:,1,1]
        skew = np.abs(Zxx + Zyy) / np.abs(Zxy - Zyx)
        valid_ratio = np.sum(skew < skew_threshold) / len(skew)

        # Consider site 1D if 80% of frequencies have skew < threshold
        return valid_ratio >= 0.8, valid_ratio, np.nanmean(skew)
    except Exception as e:
        print(f"Error reading {edi_path}: {e}")
        return False, 0, np.nan

# ---- MAIN ----
data_folder = "../data/Barotse Basin, Western Province, Zambia"  # change this to your directory path
results = []

for file in os.listdir(data_folder):
    if file.endswith(".edi"):
        path = os.path.join(data_folder, file)
        is1D, ratio, mean_skew = is_1d_site(path)
        results.append((file, is1D, ratio, mean_skew))

# ---- Summary ----
print("\n=== 1D Site Screening Summary ===")
for f, is1D, ratio, mean_skew in results:
    status = "1D-LIKE" if is1D else "Not 1D"
    print(f"{f:25s} | {status:8s} | skew<0.2 ratio={ratio:.2f} | mean_skew={mean_skew:.2f}")

# Optional: Save to CSV
import pandas as pd
df = pd.DataFrame(results, columns=["Site", "Is1D", "ValidRatio", "MeanSkew"])
df.to_csv("1D_site_check.csv", index=False)



=== 1D Site Screening Summary ===
PRIDE.ZAM001A_RH_ZAM00.2011-2012.edi | 1D-LIKE  | skew<0.2 ratio=0.93 | mean_skew=0.08
PRIDE.ZAM002A_RH_ZAM00.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.49 | mean_skew=0.37
PRIDE.ZAM003A_RH_ZAM00.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.54 | mean_skew=0.25
PRIDE.ZAM004A_RH_ZAM00.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.13 | mean_skew=1.12
PRIDE.ZAM006A_RH_ZAM00.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.57 | mean_skew=0.45
PRIDE.ZAM007A_RH_ZAM01.2011-2012.edi | 1D-LIKE  | skew<0.2 ratio=0.92 | mean_skew=0.16
PRIDE.ZAM008A_RH_ZAM01.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.33 | mean_skew=0.31
PRIDE.ZAM009A_RH_ZAM01.2011-2012.edi | 1D-LIKE  | skew<0.2 ratio=0.96 | mean_skew=0.08
PRIDE.ZAM010A_RH_ZAM00.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.80 | mean_skew=0.22
PRIDE.ZAM011A_RH_ZAM00.2011-2012.edi | 1D-LIKE  | skew<0.2 ratio=0.93 | mean_skew=0.10
PRIDE.ZAM012A_RH_ZAM00.2011-2012.edi | Not 1D   | skew<0.2 ratio=0.76 | mean_skew=0.22
PRIDE.ZA

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from mt_metadata.transfer_functions import TF

def get_sitename(edi_path):
    with open(edi_path, "r", errors="ignore") as f:
        for line in f:
            if "SITENAME" in line:
                return line.split("=")[1].strip().strip('"')
    return os.path.basename(edi_path).split(".")[0]

def sanitize_edi(edi_path):
    """
    Clean EDI file text before reading:
    - Replace '**', '***', '****' → '-9999'
    - Replace isolated ' 0.0 ' inside data sections → '-9999'
    """
    with open(edi_path, "r", errors="ignore") as f:
        lines = f.readlines()

    cleaned = []
    for line in lines:
        if line.strip().startswith(">!"):  # keep comments
            cleaned.append(line)
            continue

       
        line = re.sub(r"\*+", " -9999 ", line)
        line = re.sub(r"(?<![0-9eE\+\-])0\.0(?![0-9eE\+\-])", " -9999 ", line)
        cleaned.append(line)

    return "".join(cleaned)


def compute_skew(edi_path):
    try:
        txt = sanitize_edi(edi_path)
        tmpfile = "_tmp_clean.edi"
        with open(tmpfile, "w", errors="ignore") as f:
            f.write(txt)

        tf = TF()
        tf.read(fn=tmpfile)
        Z = np.array(tf.impedance.data, dtype=float)
        Z[Z == -9999] = np.nan
        Zxx, Zxy, Zyx, Zyy = Z[:,0,0], Z[:,0,1], Z[:,1,0], Z[:,1,1]
        skew = np.abs(Zxx + Zyy) / np.abs(Zxy - Zyx)
        skew = skew[np.isfinite(skew)]

        if len(skew) == 0:
            return np.nan
        return np.nanmean(skew)
    except Exception:
        return np.nan

folder = "../data/all_station_EDI_data"
records, invalid_counter = [], {}

for f in os.listdir(folder):
    if f.lower().endswith(".edi"):
        path = os.path.join(folder, f)
        site = get_sitename(path)
        val = compute_skew(path)
        if np.isnan(val):
            invalid_counter[site] = invalid_counter.get(site, 0) + 1
        records.append((site, f, val))

df = pd.DataFrame(records, columns=["Site", "File", "MeanSkew"])

def ratio_low_skew(skews):
    valid = skews[~np.isnan(skews)]
    if len(valid) == 0:
        return 0
    return np.sum(valid < 0.2) / len(valid)

site_summary = (
    df.groupby("Site")["MeanSkew"]
    .agg(["mean", ratio_low_skew])
    .reset_index()
)

site_summary = (
    df.groupby("Site", as_index=False)["MeanSkew"]
      .mean()
      .rename(columns={"MeanSkew": "MeanSkew"})
)


site_summary["Is1D_like"] = site_summary["MeanSkew"] < 0.2

# sort best (lowest skew) to worst (highest skew)
site_summary = site_summary.sort_values("MeanSkew", ascending=True).reset_index(drop=True)

print(site_summary.to_string(index=False))

# save outputs
site_summary.to_csv("site_skew_summary.csv", index=False)
pd.Series(invalid_counter, name="IgnoredCount").to_csv("ignored_counts.csv")


C:\Users\shail\AppData\Local\Temp\ipykernel_28320\476579746.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  Z = np.array(tf.impedance.data, dtype=float)
C:\Users\shail\AppData\Local\Temp\ipykernel_28320\476579746.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  Z = np.array(tf.impedance.data, dtype=float)
C:\Users\shail\AppData\Local\Temp\ipykernel_28320\476579746.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  Z = np.array(tf.impedance.data, dtype=float)
C:\Users\shail\AppData\Local\Temp\ipykernel_28320\476579746.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  Z = np.array(tf.impedance.data, dtype=float)
C:\Users\shail\AppData\Local\Temp\ipykernel_28320\476579746.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  Z = np.array(tf.impedance.data, dtype=float)
C:\Users\shail\AppData\Local\Temp\ipykernel_28320\47657

                                   Site  MeanSkew  Is1D_like
        Kang to Ghazi, Botswana, Africa  0.103346       True
               Central Botswana, Africa  0.110974       True
              Kweneng, Botswana, Africa  0.148348       True
    Makgadikgadi Park, Botswana, Africa  0.149585       True
Kalahari Game Reserve, Botswana, Africa  0.220361      False
               Smithfield, South Africa  0.233682      False
                  Weiveld, South Africa  0.238262      False
            Coffiefontain, South Africa  0.245331      False
               Tshane, Botswana, Africa  0.290063      False
                Orapa, Botswana, Africa  0.308026      False
         Hardap Region, Namibia, Africa  0.488457      False
               Northern Namibia, Africa  0.516205      False
        Okavango Lake, Botswana, Africa  0.595121      False
                     NE Namibia, Africa  0.642600      False
                        Namibia, Africa  0.723177      False
              Windhoek, 

C:\Users\shail\AppData\Local\Temp\ipykernel_28320\476579746.py:46: ComplexWarning: Casting complex values to real discards the imaginary part
  Z = np.array(tf.impedance.data, dtype=float)


In [23]:
len(records)

784